# Notebook 4 — Production Line (Serial Tandem Queues)

**Series:** DES Playground · Discrete Event Simulation in SimPy  
**Prerequisites:** Notebooks 1–3

**What you'll learn:**
- Build a **multi-station production line** where entities pass through stations in sequence
- Identify the **bottleneck** — the station with the highest utilisation
- Understand how the bottleneck caps **system throughput**
- Relieve the bottleneck and observe the cascade effect
- Use per-station metrics to guide capacity decisions

---

## 1 · Background — Serial Queueing Networks

A production line is a **tandem** (serial) queueing network: each entity
passes through Station 1, then Station 2, then Station 3, in order.

```
[Arrivals] → [Station 1] → [Station 2] → [Station 3] → [Departures]
   λ=0.18       μ₁=0.25       μ₂=0.20       μ₃=0.30
              ρ₁=0.72        ρ₂=0.90       ρ₃=0.60   ← Station 2 is bottleneck
```

Key principle: **the system's maximum throughput = the bottleneck's service rate**.  
No amount of speeding up Station 1 or 3 will push throughput above μ₂.

In [ ]:
!pip install simpy --quiet

In [ ]:
import simpy
import random
import math
import statistics
from dataclasses import dataclass, field
from typing import List, Dict
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

T_TABLE = {1:12.706,2:4.303,3:3.182,4:2.776,5:2.571,6:2.447,7:2.365,
           8:2.306,9:2.262,10:2.228,15:2.131,20:2.086,25:2.060,30:2.042}

def t_crit(df):
    if df in T_TABLE: return T_TABLE[df]
    if df > 30: return 1.960
    keys = sorted(T_TABLE)
    for i in range(len(keys)-1):
        lo, hi = keys[i], keys[i+1]
        if lo < df < hi:
            return T_TABLE[lo] + (T_TABLE[hi]-T_TABLE[lo])*(df-lo)/(hi-lo)
    return 2.0

def ci(values):
    n = len(values)
    if n < 2: return dict(mean=values[0] if values else 0, half=0)
    mu = statistics.mean(values)
    s  = statistics.stdev(values)
    h  = t_crit(n-1) * s / math.sqrt(n)
    return dict(mean=mu, half=h, lo=mu-h, hi=mu+h)

print('Setup complete.')

## 2 · Station Class with Per-Station Metrics

In [ ]:
class Station:
    """Single server with time-weighted Lq and ρ̂, plus entity-cohort Wq and W."""
    def __init__(self, env: simpy.Environment, name: str, capacity: int = 1):
        self.env      = env
        self.name     = name
        self.resource = simpy.Resource(env, capacity=capacity)
        self.capacity = capacity
        self._last_t  = 0.0
        self._q_area  = 0.0
        self._b_area  = 0.0
        self.wait_times:    List[float] = []
        self.sojourn_times: List[float] = []
        self.arrivals   = 0
        self.departures = 0

    def _integrate(self, t):
        dt = t - self._last_t
        self._q_area += len(self.resource.queue) * dt
        self._b_area += self.resource.count * dt
        self._last_t  = t

    def tick(self): self._integrate(self.env.now)

    def metrics(self, warmup: float, horizon: float) -> Dict:
        self._integrate(horizon)
        obs = max(horizon - warmup, 1.0)
        lq  = self._q_area / obs
        rho = (self._b_area / obs) / self.capacity
        wq  = statistics.mean(self.wait_times)    if self.wait_times    else 0.0
        w   = statistics.mean(self.sojourn_times) if self.sojourn_times else 0.0
        return dict(name=self.name, lq=lq, rho=rho, wq=wq, w=w,
                    arrivals=self.arrivals, departures=self.departures)

print('Station class ready.')

## 3 · Production Line Simulation

In [ ]:
def run_production_line(
    arrival_rate: float,
    service_rates: List[float],   # one rate per station
    capacities:    List[int],     # servers per station
    horizon: float = 30_000,
    warmup:  float = 2_000,
    seed:    int   = 42,
) -> List[Dict]:
    """
    Simulate a serial production line.
    Returns a list of per-station metric dicts.
    """
    assert len(service_rates) == len(capacities), 'Must have same length'
    rng = random.Random(seed)
    env = simpy.Environment()

    stations = [
        Station(env, f'Station {i+1}', capacity=cap)
        for i, cap in enumerate(capacities)
    ]

    def entity(arrival_t: float):
        """Entity visits all stations in sequence."""
        track = arrival_t >= warmup
        total_wait    = 0.0
        total_service = 0.0

        for stn, mu in zip(stations, service_rates):
            stn.tick()
            stn.arrivals += 1
            q_enter = env.now
            with stn.resource.request() as req:
                yield req
                stn.tick()
                wait = env.now - q_enter
                svc  = rng.expovariate(mu)
                yield env.timeout(svc)
                stn.tick()
            stn.departures += 1
            if track:
                stn.wait_times.append(wait)
                stn.sojourn_times.append(wait + svc)
            total_wait    += wait
            total_service += svc

    def source():
        while True:
            yield env.timeout(rng.expovariate(arrival_rate))
            env.process(entity(env.now))

    env.process(source())
    env.run(until=horizon)

    return [stn.metrics(warmup, horizon) for stn in stations]

print('run_production_line() defined.')

## 4 · Baseline Configuration

Three stations in series.  Station 2 has the slowest service rate — it's
the **bottleneck** with the highest utilisation.

In [ ]:
ARRIVAL_RATE   = 0.18   # λ
SERVICE_RATES  = [0.25, 0.20, 0.30]  # μ₁, μ₂, μ₃
CAPACITIES     = [1, 1, 1]

print('Baseline configuration:')
print(f'  λ = {ARRIVAL_RATE}')
for i, (mu, cap) in enumerate(zip(SERVICE_RATES, CAPACITIES)):
    rho_th = ARRIVAL_RATE / (cap * mu)
    bottleneck = ' ← BOTTLENECK' if rho_th == max(ARRIVAL_RATE/(c*m) for m,c in zip(SERVICE_RATES, CAPACITIES)) else ''
    print(f'  Station {i+1}: μ={mu}, c={cap}, ρ_theory={rho_th:.3f}{bottleneck}')

print()
# Run 8 replications
N_REPS = 8
baseline_runs = [
    run_production_line(ARRIVAL_RATE, SERVICE_RATES, CAPACITIES,
                        horizon=40_000, warmup=2_000, seed=42+k*1000)
    for k in range(N_REPS)
]

n_stations = len(SERVICE_RATES)
print(f'{"Station":<12} {"ρ(sim)":>8} {"ρ(theory)":>10} {"Lq":>8} {"Wq":>8}')
print('-' * 52)
for i in range(n_stations):
    rho_th  = ARRIVAL_RATE / (CAPACITIES[i] * SERVICE_RATES[i])
    rho_ci  = ci([run[i]['rho'] for run in baseline_runs])
    lq_ci   = ci([run[i]['lq']  for run in baseline_runs])
    wq_ci   = ci([run[i]['wq']  for run in baseline_runs])
    mark = ' ←' if rho_th == max(ARRIVAL_RATE/(c*m) for m,c in zip(SERVICE_RATES, CAPACITIES)) else ''
    print(f'Station {i+1:<4} {rho_ci["mean"]:>8.3f} {rho_th:>10.3f} {lq_ci["mean"]:>8.3f} {wq_ci["mean"]:>8.2f}{mark}')

In [ ]:
station_names = [f'Station {i+1}' for i in range(n_stations)]
rho_means  = [ci([run[i]['rho'] for run in baseline_runs])['mean'] for i in range(n_stations)]
lq_means   = [ci([run[i]['lq']  for run in baseline_runs])['mean'] for i in range(n_stations)]
lq_ci_half = [ci([run[i]['lq']  for run in baseline_runs])['half'] for i in range(n_stations)]
wq_means   = [ci([run[i]['wq']  for run in baseline_runs])['mean'] for i in range(n_stations)]

colors = ['steelblue' if r < 0.85 else 'orangered' for r in rho_means]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

axes[0].bar(station_names, rho_means, color=colors, alpha=0.85)
axes[0].axhline(1.0, color='red', ls='--', lw=1.5, label='Unstable ρ=1')
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel('Utilisation ρ')
axes[0].set_title('Utilisation per station')
axes[0].legend()

axes[1].bar(station_names, lq_means, color=colors, alpha=0.85,
            yerr=lq_ci_half, error_kw=dict(capsize=5))
axes[1].set_ylabel('Mean queue length Lq')
axes[1].set_title('Queue length per station')

axes[2].bar(station_names, wq_means, color=colors, alpha=0.85)
axes[2].set_ylabel('Mean wait time Wq')
axes[2].set_title('Wait time per station')

bottleneck_patch = mpatches.Patch(color='orangered', alpha=0.85, label='Bottleneck')
normal_patch     = mpatches.Patch(color='steelblue', alpha=0.85, label='Non-bottleneck')
fig.legend(handles=[bottleneck_patch, normal_patch], loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.04))

plt.suptitle('Baseline production line — per-station metrics', fontsize=13)
plt.tight_layout()
plt.show()

## 5 · Relieving the Bottleneck

We try two interventions on Station 2 (the bottleneck):

1. **Add a second server** at Station 2 → `capacity=2`
2. **Speed up service** by 25 % → `μ₂ = 0.25`

We compare system-level throughput and total Wq.

In [ ]:
scenarios = {
    'Baseline':              (SERVICE_RATES,          CAPACITIES),
    'Add server at S2':      ([0.25, 0.20, 0.30],     [1, 2, 1]),
    'Speed up S2 (+25%)':    ([0.25, 0.25, 0.30],     [1, 1, 1]),
    'Speed up S2 (+50%)':    ([0.25, 0.30, 0.30],     [1, 1, 1]),
}

scenario_results = {}

for name, (svc_rates, caps) in scenarios.items():
    runs = [
        run_production_line(ARRIVAL_RATE, svc_rates, caps,
                            horizon=40_000, warmup=2_000, seed=42+k*1000)
        for k in range(N_REPS)
    ]
    # Aggregate across all stations per run
    total_wq_per_run = [sum(run[i]['wq'] for i in range(len(svc_rates))) for run in runs]
    max_rho_per_run  = [max(run[i]['rho'] for i in range(len(svc_rates))) for run in runs]
    scenario_results[name] = {
        'total_wq': ci(total_wq_per_run),
        'max_rho':  ci(max_rho_per_run),
        'per_station': [ci([run[i]['rho'] for run in runs]) for i in range(len(svc_rates))],
    }

print(f'{"Scenario":<30} {"Total Wq":>10} {"±CI":>6} {"Max ρ":>8}')
print('-' * 60)
for name, res in scenario_results.items():
    wq = res['total_wq']
    rh = res['max_rho']
    print(f'{name:<30} {wq["mean"]:>10.3f} {wq["half"]:>6.3f} {rh["mean"]:>8.3f}')

In [ ]:
names_s = list(scenario_results.keys())
wq_m  = [scenario_results[n]['total_wq']['mean'] for n in names_s]
wq_h  = [scenario_results[n]['total_wq']['half'] for n in names_s]
rho_m = [scenario_results[n]['max_rho']['mean']  for n in names_s]

x = np.arange(len(names_s))
fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

bars = ax1.bar(x - 0.2, wq_m, 0.35, label='Total Wq (left)',
               color='steelblue', alpha=0.8, yerr=wq_h,
               error_kw=dict(capsize=4))
ax2.bar(x + 0.2, rho_m, 0.35, label='Max ρ (right)', color='coral', alpha=0.7)

ax1.set_xticks(x)
ax1.set_xticklabels(names_s, rotation=12, ha='right')
ax1.set_ylabel('Total wait time Wq (all stations)', color='steelblue')
ax2.set_ylabel('Max station utilisation ρ', color='coral')
ax1.set_title('Bottleneck intervention — impact on system performance')

lines1, l1 = ax1.get_legend_handles_labels()
lines2, l2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, l1+l2, loc='upper right')
plt.tight_layout()
plt.show()

## 6 · The Cascade Effect — Fixing One Bottleneck Reveals the Next

When we add a server at Station 2, its utilisation drops — but then Station 1
or 3 may become the new bottleneck.  Let's trace utilisation as we upgrade
stations one by one.

In [ ]:
cascade_scenarios = {
    'Original (S2 bottleneck)':       ([0.25, 0.20, 0.30], [1, 1, 1]),
    'Fix S2 → c=2':                   ([0.25, 0.20, 0.30], [1, 2, 1]),
    'Fix S2 → c=2, fix S1 → μ=0.30': ([0.30, 0.20, 0.30], [1, 2, 1]),
}

fig, axes = plt.subplots(1, len(cascade_scenarios), figsize=(13, 5), sharey=True)

for ax, (name, (svc_rates, caps)) in zip(axes, cascade_scenarios.items()):
    runs = [
        run_production_line(ARRIVAL_RATE, svc_rates, caps,
                            horizon=30_000, warmup=2_000, seed=42+k*1000)
        for k in range(6)
    ]
    rho_per_stn = [ci([run[i]['rho'] for run in runs])['mean'] for i in range(len(svc_rates))]
    max_rho = max(rho_per_stn)
    colors  = ['orangered' if r == max_rho else 'steelblue' for r in rho_per_stn]

    ax.bar([f'S{i+1}' for i in range(len(svc_rates))], rho_per_stn, color=colors, alpha=0.8)
    ax.axhline(1.0, color='red', ls='--', lw=1, alpha=0.5)
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('Station')
    if ax == axes[0]: ax.set_ylabel('Utilisation ρ')
    ax.set_ylim(0, 1.05)

plt.suptitle('Cascade effect — fixing one bottleneck reveals the next', fontsize=12)
plt.tight_layout()
plt.show()

## 7 · Summary

| Concept | Detail |
|---------|--------|
| **Bottleneck** | Station with highest ρ; caps system throughput |
| **Throughput** | Min(all μᵢ × cᵢ) — the bottleneck service rate |
| **Fix bottleneck first** | Improving non-bottleneck stations has near-zero effect |
| **Cascade effect** | After fixing bottleneck, the *next* slowest becomes the new one |
| **Metric to watch** | Per-station ρ and Lq — not just total system wait |

**Next:** [Notebook 5](05_what_if_analysis.ipynb) — Scenario comparison and cost-benefit analysis.